In [1]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb

In [2]:
DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_big_best_customers_25c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()

In [3]:
target = df.groupby(["customer_id", "product_id"])["tn"].shift(-2)
# obtengo las 20 columnas con mayor correlación con el target
correlation = df[numeric_cols].corrwith(target).abs().sort_values(ascending=False)
top_20_cols = correlation.head(20).index.tolist()
print("Top 20 columns with highest correlation to target:")
print(top_20_cols)

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Top 20 columns with highest correlation to target:
['tn_wavelet_0_mean_lag_11', 'tn_wavelet_0_mean_lag_8', 'tn_wavelet_0_mean_lag_15', 'tn_wavelet_0_mean', 'tn_wavelet_0_mean_lag_2', 'tn_wavelet_0_mean_lag_1', 'tn_wavelet_0_mean_lag_3', 'tn_rolling_mean_12', 'tn_wavelet_0_mean_lag_6', 'tn_rolling_mean_12_lag_1', 'tn_wavelet_0_mean_lag_20', 'tn_rolling_mean_12_lag_2', 'tn_rolling_mean_12_lag_3', 'tn_rolling_mean_24', 'tn_rolling_mean_6_lag_6', 'tn_rolling_mean_24_lag_1', 'tn_wavelet_0_max_lag_11', 'tn_wavelet_0_max_lag_15', 'tn_wavelet_0_max', 'tn_wavelet_0_max_lag_2']


In [4]:
transformations = {
    "tn": [
        r"tn$",
        r"cust_request_qty_per_tn$",
        r"tn_lag_*",
        r"tn_rolling_mean_*",
        r"tn_rolling_max_*",
        r"tn_rolling_min_*",
        r"tn_.*_vendidas$",
        r"tn_agg*",
        r"tn_wavelet_*",
    ]
    + [r"stock_final$"]
    + [r"cust_request_tn_minus_tn$"]
    + [r"tn_diff_*"],
    "cust_request_qty": [
        r"cust_request_qty$",
        r"cust_request_qty_lag_*",
        r"cust_request_qty_rolling_mean_*",
        r"cust_request_qty_rolling_max_*",
        r"cust_request_qty_rolling_min_*",
        r"cust_request_qty_.*_vendidas$",
        r"cust_request_qty_agg*",
        r"cust_request_qty_wavelet_*",
    ]
    + [r"cust_request_qty_diff_*"],
}

# busco todas las columnas que empiezan con prod_ y agrego key y valor en transformation
for col in numeric_cols:
    if col.startswith("prod_"):
        transformations[col] = [r"{}$".format(col)]
transformations

{'tn': ['tn$',
  'cust_request_qty_per_tn$',
  'tn_lag_*',
  'tn_rolling_mean_*',
  'tn_rolling_max_*',
  'tn_rolling_min_*',
  'tn_.*_vendidas$',
  'tn_agg*',
  'tn_wavelet_*',
  'stock_final$',
  'cust_request_tn_minus_tn$',
  'tn_diff_*'],
 'cust_request_qty': ['cust_request_qty$',
  'cust_request_qty_lag_*',
  'cust_request_qty_rolling_mean_*',
  'cust_request_qty_rolling_max_*',
  'cust_request_qty_rolling_min_*',
  'cust_request_qty_.*_vendidas$',
  'cust_request_qty_agg*',
  'cust_request_qty_wavelet_*',
  'cust_request_qty_diff_*'],
 'prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_8': ['prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_8$'],
 'prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_15': ['prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_15$'],
 'prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean': ['prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean$'],
 'prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_2': ['prod_tn_wavelet_0_mean_lag_11_x_tn_

In [5]:
# hago el scaling que sera column/std(col_ref) para cada columna

# primero calculo el std por product_id y customer_id por cada key de transformation
# gruped_std tiene doble indice product_id y customer_id
prod_stats = df.groupby(["product_id"])[
    list(transformations.keys())
].agg(["std"])
prod_stats.columns = [
    f"{col[0]}_{col[1]}" for col in prod_stats.columns
]  # renombro las columnas para que no tengan tupla

prod_stats = prod_stats.reset_index()
# hago un replace nan por 0 y clipeo a (1, max) para evitar inf

df = df.set_index(['product_id'])
prod_stats = prod_stats.set_index(['product_id'])
# supress performance warnings
from pandas.errors import PerformanceWarning
import warnings
warnings.simplefilter(action="ignore", category=PerformanceWarning)

for trainer, regex_cols in transformations.items():
    for col in regex_cols:
        matching_cols = [c for c in numeric_cols if re.match(col, c)]
        if not matching_cols:
            continue
        print(f"Processing trainer: {trainer} with columns: {matching_cols}")
        for col in matching_cols:
            std_col = prod_stats[trainer + "_std"].clip(lower=1, upper=None)
            # Alinear por índice, sin merge
            df[f"{col}_scaled"] = (df[col] / std_col).replace([np.inf, -np.inf], np.nan)
            # Opcional: fillna(0) si querés
            # df[f"{col}_scaled"] = df[f"{col}_scaled"].fillna(0)

df = df.reset_index()

Processing trainer: tn with columns: ['tn']
Processing trainer: tn with columns: ['tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_6', 'tn_lag_8', 'tn_lag_11', 'tn_lag_15', 'tn_lag_20']
Processing trainer: tn with columns: ['tn_rolling_mean_6', 'tn_rolling_mean_12', 'tn_rolling_mean_24', 'tn_rolling_mean_12_lag_1', 'tn_rolling_mean_12_lag_2', 'tn_rolling_mean_12_lag_3', 'tn_rolling_mean_12_lag_6', 'tn_rolling_mean_12_lag_8', 'tn_rolling_mean_12_lag_11', 'tn_rolling_mean_12_lag_15', 'tn_rolling_mean_12_lag_20', 'tn_rolling_mean_24_lag_1', 'tn_rolling_mean_24_lag_2', 'tn_rolling_mean_24_lag_3', 'tn_rolling_mean_24_lag_6', 'tn_rolling_mean_24_lag_8', 'tn_rolling_mean_24_lag_11', 'tn_rolling_mean_6_lag_1', 'tn_rolling_mean_6_lag_2', 'tn_rolling_mean_6_lag_3', 'tn_rolling_mean_6_lag_6', 'tn_rolling_mean_6_lag_8', 'tn_rolling_mean_6_lag_11', 'tn_rolling_mean_6_lag_15', 'tn_rolling_mean_6_lag_20']
Processing trainer: tn with columns: ['tn_rolling_max_6', 'tn_rolling_max_12', 'tn_rolling_max_24', '

In [6]:
df[["tn", "tn_scaled"]].describe()

,tn,tn_scaled
count,819572.000000,819572.000000
mean,1.616684,0.282059
std,9.578342,0.811844
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.023370,0.017780
75%,0.454495,0.197374
max,751.269836,18.664850


In [7]:
prod_stats

,tn_std,cust_request_qty_std,prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_8_std,prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_15_std,prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_std,prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_2_std,prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_1_std,prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_3_std,prod_tn_wavelet_0_mean_lag_11_x_tn_rolling_mean_12_std,prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_6_std,...,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_11_std,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_15_std,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_std,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_2_std,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_lag_15_std,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_std,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_lag_2_std,prod_tn_wavelet_0_max_lag_15_x_tn_wavelet_0_max_std,prod_tn_wavelet_0_max_lag_15_x_tn_wavelet_0_max_lag_2_std,prod_tn_wavelet_0_max_x_tn_wavelet_0_max_lag_2_std
product_id,,,,,,,,,,,,,,,,,,,,,
20001,77.989960,44.567962,48721.880136,48729.031447,48721.880136,48721.880136,48721.880136,48721.880136,26484.231406,48721.880136,...,39943.103360,39943.103360,39943.103360,39943.103360,114654.316853,114637.490567,114637.490567,114654.316853,114654.316853,114614.119273
20002,66.334320,43.951545,38671.013630,38676.689693,38671.013630,38671.013630,38671.013630,38671.013630,21115.673394,38671.013630,...,37329.835219,37329.835219,37329.835219,37329.835219,142719.274607,142698.329600,142698.329600,142719.274607,142719.274607,142669.237507
20003,58.343838,48.722399,48825.006561,48832.173009,48825.006561,48825.006561,48825.006561,48825.006561,25247.132835,48825.006561,...,42090.832748,42090.832748,42090.832748,42090.832748,134947.199419,134927.395016,134927.395016,134947.199419,134947.199419,134899.887194
20004,69.092476,50.374771,79379.236604,79390.887747,79379.236604,79379.236604,79379.236604,79379.236604,41511.504341,79379.236604,...,73616.252201,73616.252201,73616.252201,73616.252201,249086.937717,249050.382546,249050.382546,249086.937717,249086.937717,248999.608323
20005,66.677444,42.132072,68743.242883,68753.332894,68743.242883,68743.242883,68743.242883,68743.242883,34437.556692,68743.242883,...,49601.334892,49601.334892,49601.334892,49601.334892,144758.638207,144737.393910,144737.393910,144758.638207,144758.638207,144707.886111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21295,0.001371,0.196116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21296,0.001277,0.196116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21297,0.001136,0.196116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
prod_stats = prod_stats.reset_index()
# creo el target
final_test_df = df[df["date_id"] == df["date_id"].max()]

df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)
# elimino las rows donde target es nan
df = df[df["target"].notna()]
# hago el scaling del target por tn_std
df = df.merge(
    prod_stats[["product_id", "tn_std"]],
    on=["product_id"],
    how="left",
)
df["target_scaled"] = (df["target"] / df["tn_std"]).fillna(0)
# replace inf and -inf with 0
df["target_scaled"] = df["target_scaled"].replace([np.inf, -np.inf], 0)
# replace nan with 0

#submision test_df es el maximo date_id de df

df.drop(columns=["tn_std"], inplace=True)


In [9]:
prod_stats[["tn_std", "product_id"]]

,tn_std,product_id
0,77.989960,20001
1,66.334320,20002
2,58.343838,20003
3,69.092476,20004
4,66.677444,20005
...,...,...
1228,0.001371,21295
1229,0.001277,21296
1230,0.001136,21297
1231,0.001124,21298


In [10]:
df["target_scaled"].describe()

count    755976.000000
mean          0.371147
std           0.981632
min           0.000000
25%           0.000000
50%           0.036773
75%           0.285938
max          20.298187
Name: target_scaled, dtype: float64

In [11]:
# reemplazar todos los valores inf y -inf por np.nan
df.replace([np.inf, -np.inf], np.nan, inplace=True) 

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import BaseCrossValidator

class CustomTimeSeriesSplitter(BaseCrossValidator):
    def __init__(self, subsample_prop=0.5, test_months=1, random_state=None, random=True, quantiles=10):
        assert 0 < subsample_prop <= 1, "subsample_prop must be in (0, 1]"
        self.subsample_prop = subsample_prop
        self.test_months = test_months
        self.random_state = np.random.RandomState(random_state)
        self._sampled_series = None
        self.random = random
        self.quantiles = quantiles

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.test_months

    def split(self, X, y=None, groups=None):
        df = X.reset_index(drop=True)
        max_date = df['date_id'].max()
        pair_col = ['product_id', 'customer_id']

        # Calcular tn total y asignar deciles
        total_tn = (
            df.groupby(pair_col)['tn'].sum()
            .reset_index(name='total_tn')
            .sort_values('total_tn', ascending=False)
            .reset_index(drop=True)
        )
        total_tn['quantile'] = pd.qcut(total_tn.index, self.quantiles, labels=False)

        # Samplear series por quantil
        if not self._sampled_series:
            sampled_series = []
            for q in range(10):
                group = total_tn[total_tn['quantile'] == q]
                n = max(1, int(len(group) * self.subsample_prop))
                if self.random:
                    # Si es aleatorio, usar random_state
                    sampled = group.sample(n=n, random_state=self.random_state)
                else:
                    # los primeros N
                    sampled = group.head(n)
                sampled_series.append(sampled)
            self._sampled_series = sampled_series
        else:
            sampled_series = self._sampled_series

        sampled_series_df = pd.concat(sampled_series, ignore_index=True)

        # Marcar las series seleccionadas (más eficiente que set + apply)
        sampled_df = df.merge(sampled_series_df[pair_col], on=pair_col, how='inner')

        for i in range(self.test_months):
            test_date = max_date - i
            test_mask = df['date_id'] == test_date
            test_idx = df.index[test_mask].tolist()

            # Entrenamiento: solo series seleccionadas y fechas anteriores
            train_mask = (sampled_df['date_id'] < test_date)
            train_idx = sampled_df.index[train_mask].tolist()

            yield train_idx, test_idx

class SimpleLastDateSplitter(BaseCrossValidator):
    """Split: test = date_id máximo, train = resto. Sin copias innecesarias."""
    def get_n_splits(self, X=None, y=None, groups=None):
        return 1

    def split(self, X, y=None, groups=None):
        # No copies, solo uso la referencia
        max_date = X['date_id'].max()
        test_mask = X['date_id'] == max_date
        
        # Obtener posiciones enteras (para .iloc) en lugar de índices del DataFrame
        test_idx = np.where(test_mask)[0]
        train_idx = np.where(~test_mask)[0]
        
        yield train_idx, test_idx

In [13]:
# Reemplazo de inf por nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)



# Transformar object a category (menos claves)
columns = df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        df[col] = df[col].astype("category")

# Eliminar columnas datetime innecesarias
datetime_cols = df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col != "date_id":
        df.drop(columns=[col], inplace=True)

# Crear splitter con 20% y 2 meses

# Columnas a dropear
drop_cols = ["fecha", "target_scaled", "target", "date_id"]

In [14]:

class LGBCustomMetric:
    def __init__(self, test_df, tn_std, product_ids):
        # Guardamos referencias completas sin filtrar
        self.test_df = test_df[["product_id", "target"]].copy()
        self.tn_std = tn_std[["product_id", "tn_std"]].copy()
        self.product_ids = set(product_ids)

    def __call__(self, preds, train_data):
        # Recrear la evaluación exactamente como en la función objective
        test_df_eval = self.test_df.copy()
        test_df_eval["predictions"] = preds
        test_df_eval = test_df_eval.groupby("product_id").agg({
            "predictions": "sum",
            "target": "sum"
        }).reset_index()
        test_df_eval = test_df_eval[test_df_eval["product_id"].isin(self.product_ids)]
        test_df_eval = test_df_eval.merge(
            self.tn_std,
            on=["product_id"],
            how="left",
        )
        test_df_eval["predictions"] *= test_df_eval["tn_std"]
        total_error = np.sum(np.abs(test_df_eval["predictions"] - test_df_eval["target"])) / test_df_eval["target"].sum()
        return "total_error", total_error, False

In [16]:
import optuna
import numpy as np
import lightgbm as lgb

splitter = SimpleLastDateSplitter()
for fold, (train_idx, test_idx) in enumerate(splitter.split(df)):
    # HACER COPIAS para evitar modificar los datos originales
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

train_df = train_df.groupby(["product_id", "customer_id"]).filter(lambda x: len(x) >= 12)


print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}, Fold: {fold + 1}")

train = lgb.Dataset(
    train_df.drop(columns=[col for col in drop_cols if col in train_df.columns]),
    label=train_df["target_scaled"],
    categorical_feature="auto",
    free_raw_data=True,
    weight=np.log1p(train_df["tn"]).clip(lower=1),
)

dtest = lgb.Dataset(
    test_df.drop(columns=[col for col in drop_cols if col in test_df.columns]),
    label=test_df["target_scaled"],
    categorical_feature="auto",
    free_raw_data=True,
)

custom_metric = LGBCustomMetric(
    test_df=test_df,
    tn_std=prod_stats[["product_id", "tn_std"]],
    product_ids=product_ids,
)

def objective(trial):
    total_errors = []
    num_iterations_list = []

    # Hiperparámetros a optimizar
    params = {
        #'objective': 'tweedie',
        'objective': 'regression', # tweedie no funciona bien con cuda
        'boosting_type': 'gbdt',
        'force_row_wise': True,
        'verbose': -1,
        'metric': 'None',
        'extra_trees': True,
        'first_metric_only': True,
        "max_depth": -1,
        #"device": "cuda",
        "feature_pre_filter": False,
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.1),
        'num_leaves': trial.suggest_int("num_leaves", 15, 256),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        'feature_fraction': trial.suggest_float("feature_fraction", 0.1, 1.0),
        'bagging_fraction': trial.suggest_float("bagging_fraction", 0.5, 1.0),
        'bagging_freq': trial.suggest_int("bagging_freq", 1, 10),
    }




    model = lgb.train(
        params=params,
        train_set=train,
        num_boost_round=9999,
        feval=custom_metric,
        valid_sets=[dtest],
        callbacks=[
            lgb.early_stopping(int(400 + 4 / params["learning_rate"]), first_metric_only=True),
            lgb.log_evaluation(period=100)
        ],
    )

    # Usar una copia separada para las predicciones finales
    test_df_eval = test_df.copy()
    predictions = model.predict(
        test_df_eval.drop(columns=[col for col in drop_cols if col in test_df_eval.columns]),
        num_iteration=model.best_iteration,
    )
    test_df_eval["predictions"] = predictions

    test_df_eval = test_df_eval.merge(
        prod_stats[["product_id", "tn_std"]],
        on=["product_id"],
        how="left",
    )
    test_df_eval["predictions"] *= test_df_eval["tn_std"]

    test_df_grouped = test_df_eval.groupby("product_id").agg({
        "predictions": "sum",
        "target": "sum"
    }).reset_index()
    test_df_grouped = test_df_grouped[test_df_grouped["product_id"].isin(product_ids)]
    abs_error = np.abs(test_df_grouped["predictions"] - test_df_grouped["target"])
    total_error = np.sum(abs_error) / np.sum(test_df_grouped["target"])

    total_errors.append(total_error)
    num_iterations_list.append(model.best_iteration)

    avg_error = np.mean(total_errors)
    trial.set_user_attr("avg_num_iterations", np.mean(num_iterations_list))
    return avg_error

# Ejecutar la optimización con Optuna
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=24),
    study_name="exp_pipeline_lgb_60_trial_regression_2",
    storage="sqlite:///optuna_study.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=60)

Train shape: (698152, 1070), Test shape: (23998, 1070), Fold: 1


[I 2025-07-13 13:47:47,119] A new study created in RDB with name: exp_pipeline_lgb_60_trial_regression_2


Training until validation scores don't improve for 441 rounds
[100]	valid_0's total_error: 0.258244
[200]	valid_0's total_error: 0.253386
[300]	valid_0's total_error: 0.253318
[400]	valid_0's total_error: 0.251899


[W 2025-07-13 13:55:22,037] Trial 0 failed with parameters: {'learning_rate': 0.09640155730023267, 'num_leaves': 184, 'lambda_l1': 9.972536479100224, 'lambda_l2': 9.563254159778875e-07, 'min_data_in_leaf': 37, 'feature_fraction': 0.7658568911884933, 'bagging_fraction': 0.9982278625445484, 'bagging_freq': 4} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_288803/1408039211.py", line 67, in objective
    model = lgb.train(
            ^^^^^^^^^^
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/lightgbm/engine.py", line 322, in train
    booster.update(fobj=fobj)
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/lightgbm/basic.py", line 4155, in update
    _LIB.LGBM_BoosterUpdateOneIter(
Key

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params
best_params.update({
    'objective': 'regression', # tweedie no funciona bien con cuda
    'boosting_type': 'gbdt',
    'force_row_wise': True,
    'verbose': -1,
    'metric': 'None',
    'extra_trees': True,
    'first_metric_only': True,
    "max_depth": -1,
    "device": "cuda",
    "feature_pre_filter": False,
})


avg_best_iter = int(study.best_trial.user_attrs["avg_num_iterations"])
print(f"🔍 Mejor total_error: {study.best_value:.5f}")
print(f"🏁 Mejor iteración promedio: {avg_best_iter}")
print(f"📊 Mejor conjunto de parámetros: {best_params}")

# Entrenar modelo final con todo el dataset
final_df = df.groupby(["product_id", "customer_id"]).filter(lambda x: len(x) >= 12)
final_train = lgb.Dataset(
    final_df.drop(columns=[col for col in drop_cols if col in final_df.columns]),
    label=final_df["target_scaled"],
    categorical_feature="auto",
    #weight=(final_df["tn"] + 0.1),
)

final_model = lgb.train(
    params=best_params,
    train_set=final_train,
    num_boost_round=avg_best_iter,
    callbacks=[lgb.log_evaluation(period=10)],
)

🔍 Mejor total_error: 0.23013
🏁 Mejor iteración promedio: 1537
📊 Mejor conjunto de parámetros: {'tweedie_variance_power': 1.8680138426687347, 'learning_rate': 0.07295608449546184, 'num_leaves': 256, 'lambda_l1': 2.2006730056278445, 'lambda_l2': 3.6105635460300163, 'min_data_in_leaf': 74, 'feature_fraction': 0.9968101525801872, 'bagging_fraction': 0.6581734888953041, 'bagging_freq': 2, 'objective': 'tweedie', 'boosting_type': 'gbdt', 'force_row_wise': True, 'metric': 'None', 'extra_trees': True, 'first_metric_only': True, 'max_depth': -1, 'device': 'cuda', 'feature_pre_filter': False}


In [17]:
features = final_train.feature_name
# replace spaces in final_test_df columns names by _ (underscore)
final_test_df.columns = final_test_df.columns.str.replace(" ", "_", regex=False)

# Reemplazo de inf por nan
final_test_df.replace([np.inf, -np.inf], np.nan, inplace=True)



# Transformar object a category (menos claves)
columns = final_test_df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        final_test_df[col] = final_test_df[col].astype("category")

# Eliminar columnas datetime innecesarias
datetime_cols = final_test_df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col != "date_id":
        final_test_df.drop(columns=[col], inplace=True)



final_predictions = final_model.predict(final_test_df[features])
final_test_df["predictions"] = final_predictions
final_test_df = final_test_df.merge(
    prod_stats[["product_id", "tn_std"]],
    on=["product_id"],
    how="left",
)
final_test_df["predictions"] = final_test_df["predictions"] * final_test_df["tn_std"]
# agrupo por product_id y customer_id
final_test_df_grouped = final_test_df.groupby("product_id").agg({
    "predictions": "sum",
}).reset_index()
final_test_df_grouped = final_test_df_grouped[final_test_df_grouped["product_id"].isin(product_ids)]
submission = final_test_df_grouped[["product_id", "predictions"]].copy()
submission.rename(columns={"predictions": "tn"}, inplace=True)
submission["tn"] = submission["tn"].clip(lower=0)  # Asegurar que no haya valores negativos
submission.to_csv("submission_lgb.csv", index=False)
submission

,product_id,tn
0,20001,1259.909774
1,20002,1519.596494
2,20003,757.721908
3,20004,754.702732
4,20005,622.642749
...,...,...
920,21263,0.004603
922,21265,0.106636
923,21266,0.111480
924,21267,0.002821
